# 03. Data Profiling — Automated Dataset Auditing & Data Dictionaries

Data Profiling systematically audits raw data to evaluate its structure, statistical content, cardinality, and data health before pipeline ingestion.

---

### Key Profiling Metrics
- **Completeness**: Proportion of non-null values.
- **Uniqueness**: Cardinality of values vs total row count.
- **Validity**: Adherence to domain ranges, format regexes, and data types.
- **Distributional Skew**: Asymmetry and outlier tendencies.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

# Generate profiling test dataset
np.random.seed(42)
n = 2000

df_profile = pd.DataFrame({
    'employee_id': [f"EMP_{1000 + i}" for i in range(n)],
    'department': np.random.choice(['Engineering', 'Marketing', 'Sales', 'Finance', 'HR', None], size=n, p=[0.35, 0.2, 0.2, 0.15, 0.08, 0.02]),
    'salary': np.random.lognormal(mean=11.2, sigma=0.45, size=n).round(2),
    'years_experience': np.random.exponential(scale=5.0, size=n).round(1),
    'performance_rating': np.random.choice([1, 2, 3, 4, 5], size=n, p=[0.05, 0.15, 0.50, 0.20, 0.10]),
    'is_remote': np.random.choice([True, False], size=n, p=[0.6, 0.4])
})

# Inject dirty records
df_profile.loc[df_profile['salary'] > 200000, 'salary'] = np.nan
print("Profiling Target Dataset Initialized. Rows:", len(df_profile))

Profiling Target Dataset Initialized. Rows: 2000

## 1. Building an Enterprise Automated Data Profiler

A comprehensive profiling engine extracts structural metadata, missingness, cardinalities, uniqueness ratios, and distribution metrics into a consolidated table.

In [1]:
def profile_dataframe(df):
    """Generates an executive statistical profiling report for any tabular dataframe."""
    profile_data = []
    total_rows = len(df)
    
    for col in df.columns:
        s = df[col]
        dtype = str(s.dtype)
        null_count = s.isna().sum()
        null_pct = (null_count / total_rows) * 100
        unique_count = s.nunique(dropna=True)
        unique_ratio = unique_count / total_rows
        
        row_dict = {
            'Feature': col,
            'Dtype': dtype,
            'Missing': null_count,
            'Missing %': f"{null_pct:.2f}%",
            'Uniques': unique_count,
            'Uniqueness Ratio': f"{unique_ratio:.3f}"
        }
        
        if pd.api.types.is_numeric_dtype(s):
            clean_s = s.dropna()
            row_dict['Mean'] = f"{clean_s.mean():,.2f}"
            row_dict['Std'] = f"{clean_s.std():,.2f}"
            row_dict['Min'] = f"{clean_s.min():,.2f}"
            row_dict['Median'] = f"{clean_s.median():,.2f}"
            row_dict['Max'] = f"{clean_s.max():,.2f}"
            row_dict['Skewness'] = f"{clean_s.skew():,.2f}"
        else:
            top_val = s.mode().iloc[0] if not s.dropna().empty else "N/A"
            row_dict['Mean'] = "-"
            row_dict['Std'] = "-"
            row_dict['Min'] = "-"
            row_dict['Median'] = f"Top: {top_val}"
            row_dict['Max'] = "-"
            row_dict['Skewness'] = "-"
            
        profile_data.append(row_dict)
        
    return pd.DataFrame(profile_data)

profile_report = profile_dataframe(df_profile)
print("=== AUTOMATED DATA PROFILING AUDIT REPORT ===")
print(profile_report.to_string(index=False))

=== AUTOMATED DATA PROFILING AUDIT REPORT ===
           Feature   Dtype  Missing Missing %  Uniques Uniqueness Ratio        Mean         Std        Min             Median          Max Skewness
       employee_id  object        0     0.00%     2000            1.000           -           -          -       Top: EMP_1000            -        -
        department  object       41     2.05%        5            0.003           -           -          -   Top: Engineering            -        -
            salary float64       14     0.70%     1986            0.993   78,412.35   24,812.10  29,145.20          74,819.50   194,520.10     1.12
  years_experience float64        0     0.00%      184            0.092        4.98        4.89       0.00               3.60        34.20     1.79
performance_rating   int64        0     0.00%        5            0.003        3.14        0.98       1.00               3.00         5.00     0.02
         is_remote    bool        0     0.00%        2           

## 2. Generating an Automated Data Dictionary
A Data Dictionary serves as the contractual documentation between engineering and business stakeholders:
- Column identifier and human-readable description
- Storage format vs logical business format
- Permissible domain ranges or enumeration values